# **Note for Colab Users**

# **Do Not Write Directly in This File, or Your Work May Disappear!**

# **Always Make a Copy Before You Start.**

How to make a copy

1. Click **File** in the top-left corner.  
> *If you cannot see menus like "File" or "Runtime", press the “v” mark in the top-right corner to show them.*

2. Choose **Save a copy in Drive**.  

3. Rename the copied file to `YOURNAMEs_FileName.ipynb`.  
> Example: if your name is Olivia → Olivias_FileName.ipynb  


---

* Check marks (✅) are not saved. They disappear if you refresh the page with Chrome's reload button.<br>  
If you stop halfway, add a text cell and write something like `SO FAR DONE`.

---

* In Colab, **previous output is reset every 30 to 90 minutes**.<br>  
Because of that, errors like `~~ is not defined` happen **very often**.

🔁 What should you do when you get a `~~ is not defined` error?

1. First, check the spelling of the variable name.<br>  
2. If the spelling is correct but the error still appears, **click that cell to select it**.<br>  
3. Click **Runtime** → **Run before** in the top-left menu.<br>  
→ This reruns **all cells before that point**.  
4. Run that cell again.

If the error still does not go away,<br>  
there may be a basic mistake in your TODO answer in an earlier cell.<br>  
Please check whether it is correct.<br>  
Or ask ChatGPT or another coding assistant for help.


# **Preparation**

In this part, we only load the content from the previous Chapter.<br>
You just need to run the code. You do not have to read it carefully.<br>
Feel free to move on.<br>


In [ ]:
# Download the file
!wget https://raw.githubusercontent.com/HayatoHongo/EveryonesAI/main/input.txt -O input.txt
# Read the downloaded input.txt file with UTF-8.
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()

# Function for displaying tensors clearly (optional)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Install the library for displaying tensors clearly
!pip install git+https://github.com/HayatoHongo/print_formatted_tensor.git
# Import the function for displaying PyTorch tensors clearly
from torch_print_tensor import print_formatted_tensor


# **Chapter 16: Increase Only the Model Size**




### **Section 1: Improving the `Trainer` Class**


Google Colab gives you a free T4 GPU. Let’s use it.


In [ ]:
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'  # Device to use (GPU or CPU)
print(device_type)

**Check Point**  
<label><input type="checkbox">Confirmed that the runtime is set to cuda GPU<br></label>  


#### Make the model bigger and try to break through the loss plateau

Last time, we trained a **very small model** with about 200,000 parameters on a **small Shakespeare dataset** (`input.txt`) of about 100,000 characters.

But no matter how much compute we poured in, the **validation loss stalled around 1.7 to 1.8**.

This state, where the loss stops going down, is called a **loss plateau**.

So this time, **we will spend our compute on making the model larger.**

In general, **larger models with more parameters can express richer patterns, so we expect the loss plateau to move lower**.


#### Visualizing training time and processed tokens

This training run takes **about 50 minutes**.
As training moves forward, the **total number of tokens processed by the model** keeps growing.

To make the training progress easier to understand,
we will record the following information at each step.

* `total_seen_tokens`, which represents the **total number of tokens used in training** up to that point
* `total_train_time`, which represents the **cumulative training time in seconds** up to that point

To store these values, we add matching variables and lists to the `Trainer` class.


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`[]`　`()`


In [ ]:
import time

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

        self.steps = []
        self.train_losses = []
        self.val_losses = []
        ########## NEW ##########
        # List that records the number of tokens processed so far during training
        self.total_seen_tokens_list = __  # TODO: FILL
        # List that records the cumulative time spent training
        self.total_train_time_list = __ # TODO: FILL
        ########## NEW ##########

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Calculate losses for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Switch back to training mode

        # Calculate and return the average loss for each dataset (train, val)
        return {split: sum(values) / len(values) for split, values in losses.items()}

<details>
<summary>Click to show/hide the answer</summary>

```python
import time

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

        self.steps = []
        self.train_losses = []
        self.val_losses = []
        ########## NEW ##########
        self.total_seen_tokens_list = []
        self.total_train_time_list = []
        ########## NEW ##########

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Calculate losses for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Switch back to training mode

        # Calculate and return the average loss for each dataset (train, val)
        return {split: sum(values) / len(values) for split, values in losses.items()}
```


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`1`　`0`　`interval_from_last_eval`　`last_eval_end_time`　`self.config.batch_size`　`step`　`append`　`logits`　`self.config.total_training_steps`　`evaluation_interval`


In [ ]:
# Continuing the Trainer class
def train(self):
    # Run train_step the number of times specified in config.
    for step in range(self.config.total_training_steps):
        # Evaluate every 100 steps, or only on the final step.
        if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
            # Exclude step==0 because last_eval_end_time is not defined yet. Exclude the final step because it may be a partial measurement.
            if step == 0 or step == self.config.total_training_steps - 1:
              tokens_per_second = None
              ########## NEW ##########
              if step == 0:
                # On the first step, no time has elapsed yet, so initialize the cumulative time.
                total_train_time = __ # TODO: FILL
              else: # step == self.config.total_training_steps - 1
                # On the final step, add the elapsed time since the last evaluation
                current_time = time.time()
                interval_from_last_eval = current_time - _____________ # TODO: FILL
                total_train_time += ______________ # TODO: FILL
              ########## NEW ##########
            else:
              current_eval_start_time = time.time()
              evaluation_interval = current_eval_start_time - last_eval_end_time
              ########## NEW ##########
              # Add the elapsed time between evaluations to the cumulative training time
              total_train_time += _____________ # TODO: FILL
              ########## NEW ##########
              tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
              tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

            eval_loss = self.evaluate()

            ########## NEW ##########
            # Calculate the number of tokens processed so far
            total_seen_tokens = ________________ * self.config.input_sequence_length * _____ # TODO: FILL
            ########## NEW ##########

            ########## NEW ##########
            # Print step, loss, processing speed (tok/s), token count, and elapsed time together
            print(
                f"step {step:05d} | "
                f"train loss {eval_loss['train']:.4f} | "
                f"val loss {eval_loss['val']:.4f} | "
                f"tok/s {int(tokens_per_second) if tokens_per_second is not None else 'None'} | "
                f"tokens {total_seen_tokens:,} | "
                f"time {total_train_time:.2f}s"
            )
            ########## NEW ##########

            self.steps.append(step)
            self.train_losses.append(eval_loss['train'])
            self.val_losses.append(eval_loss['val'])

            ########## NEW ##########
            # Add the cumulative token count and elapsed time to the log lists
            self.total_seen_tokens_list.______(total_seen_tokens) # TODO: FILL
            self.total_train_time_list.append(total_train_time)
            ########## NEW ##########

            # Record the time when this evaluation ends. The time difference from the next evaluation start becomes `evaluation_interval`.
            last_eval_end_time = time.time()

        # One training step (the main process done every time)
        train_loss = self.train_step()

<details>
<summary>Click to show/hide the answer</summary>

```python
def train(self):
    # Run train_step the number of times specified in config.
    for step in range(self.config.total_training_steps):

        # Evaluate every 100 steps, or only on the final step.
        if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
            # Exclude step==0 because last_eval_end_time is not defined yet. Exclude the final step because it may be a partial measurement.
            if step == 0 or step == self.config.total_training_steps - 1:
              tokens_per_second = None
              ########## NEW ##########
              # First step
              if step == 0:
                total_train_time = 0
              # Final step
              else:
                current_time = time.time()
                interval_from_last_eval = current_time - last_eval_end_time
                total_train_time += interval_from_last_eval
              ########## NEW ##########
            else:
              current_eval_start_time = time.time()
              evaluation_interval = current_eval_start_time - last_eval_end_time
              ########## NEW ##########
              total_train_time += evaluation_interval
              ########## NEW ##########
              tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
              tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

            eval_loss = self.evaluate()

            ########## NEW ##########
            total_seen_tokens = self.config.batch_size * self.config.input_sequence_length * step
            ########## NEW ##########

            ########## NEW ##########
            print(
                f"step {step:05d} | "
                f"train loss {eval_loss['train']:.4f} | "
                f"val loss {eval_loss['val']:.4f} | "
                f"tok/s {int(tokens_per_second) if tokens_per_second is not None else 'None'} | "
                f"tokens {total_seen_tokens:,} | "
                f"time {total_train_time:.2f}s"
            )
            ########## NEW ##########

            self.steps.append(step)
            self.train_losses.append(eval_loss['train'])
            self.val_losses.append(eval_loss['val'])
            ########## NEW ##########
            self.total_seen_tokens_list.append(total_seen_tokens)
            self.total_train_time_list.append(total_train_time)
            ########## NEW ##########

            # Record the time when this evaluation ends. The time difference from the next evaluation start becomes `evaluation_interval`.
            last_eval_end_time = time.time()

        # One training step (the main process done every time)
        train_loss = self.train_step()
```


🔧 Formula for adding or replacing a class method

```python
def new_function(self, ...):
    ...

ExistingClass.method_name = new_function
```


In [ ]:
Trainer.train = train

The `train` function has become long, and the extra `if` statements make it harder to read.
So we will refactor it to improve readability.

Specifically, we will align the "final step" with the "evaluation step" to reduce branching.

The current loop is:

```python
for step in range(self.config.total_training_steps):
```

So `step` runs from `0` to `total_training_steps - 1`. For example, if `total_training_steps = 1000` and `evaluation_frequency = 100`, the final `step` is `999`, which does not line up with the evaluation timing (multiples of 100).

---

So we increase the total steps by exactly 1, like this:

```python
for step in range(self.config.total_training_steps + 1):
```

Now the final step is included in the evaluation targets too. The code becomes simpler.

Note: this does not apply when `total_training_steps` is not a multiple of `evaluation_frequency` (for example, 1050 and 100), but it is usually fine.


In [ ]:
def new_train(self):
    # Run train_step (the number of times specified in config + 1).)
    ########## NEW ##########
    for step in range(self.config.total_training_steps+1):
    ########## NEW ##########
        # Evaluate every 100 steps.
        ########## NEW ##########
        if step % self.config.evaluation_frequency == 0:
        ########## NEW ##########

            ########## NEW ##########
            if step == 0:
              tokens_per_second = None
              total_train_time = 0
            ########## NEW ##########
            else:
              current_eval_start_time = time.time()
              evaluation_interval = current_eval_start_time - last_eval_end_time
              total_train_time += evaluation_interval
              tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
              tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

            eval_loss = self.evaluate()
            total_seen_tokens = self.config.batch_size * self.config.input_sequence_length * step

            print(
                f"step {step:05d} | "
                f"train loss {eval_loss['train']:.4f} | "
                f"val loss {eval_loss['val']:.4f} | "
                f"tok/s {int(tokens_per_second) if tokens_per_second is not None else 'None'} | "
                f"tokens {total_seen_tokens:,} | "
                f"time {total_train_time:.2f}s"
            )

            self.steps.append(step)
            self.train_losses.append(eval_loss['train'])
            self.val_losses.append(eval_loss['val'])
            self.total_seen_tokens_list.append(total_seen_tokens)
            self.total_train_time_list.append(total_train_time)

            # Record the time when this evaluation ends. The time difference from the next evaluation start becomes `evaluation_interval`.
            last_eval_end_time = time.time()

        # One training step (the main process done every time)
        train_loss = self.train_step()

🔧 Formula for adding or replacing a class method

```python
def new_function(self, ...):
    ...

ExistingClass.method_name = new_function
```

* **The method name stays the same** (you can call it with `train()`)
* **Only the contents are replaced** (the process inside `new_train` is executed)
<br>


In [ ]:
Trainer.train = # TODO: Hint: Do not overthink it. It is very simple.

**Section 1: Improving the `Trainer` Class** <label><input type="checkbox"> Mark as Done</label>


### **Section 2: Overfitting in LLMs**


Define the `Dataloader` class and the model class. This is exactly the same as Chapter 12.


In [ ]:
class DataLoader:
    def __init__(self, text, config):
        self.config = config  # Config object
        chars = sorted(list(set(text)))  # Sort the unique characters
        self.ctoi = {char: index for index, char in enumerate(chars)}
        self.itoc = {index: char for index, char in enumerate(chars)}
        self.vocab_size = len(chars)

        # Encode and convert to a tensor.
        # You need `self.` to call methods or arguments outside `__init__`.
        self.data = torch.tensor(self.encode(text), dtype=torch.long)

        # Split into training and validation data.
        # self.data is used even when no argument is specified.
        self.train_data, self.val_data = self.split_data()

    def encode(self, text):
        # Convert a string into a sequence of indices. Use self. to call other methods or arguments.
        return [self.ctoi[c] for c in text]

    def decode(self, indices):
        return ''.join([self.itoc[i] for i in indices])

    def split_data(self):
        split_index = int(0.9 * len(self.data))  # The point where 90% of the data is split for training.
        return self.data[:split_index], self.data[split_index:]

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        start_indices = torch.randint(len(data) - self.config.input_sequence_length, (self.config.batch_size,)) # Generate starting indices for extraction

        input_sequences = torch.stack([
            data[start_index:start_index + self.config.input_sequence_length]
            for start_index in start_indices
        ])
        target_sequences = torch.stack([
            data[start_index + 1:start_index + self.config.input_sequence_length + 1]
            for start_index in start_indices
        ])
        return input_sequences.to(self.config.device_type), target_sequences.to(self.config.device_type)

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Define an embedding table of vocab size x embedding dimension
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        # Get the embedding vectors for the input indices
        return self.token_embedding_table.forward(input_indices)

class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length = 8, embedding_dim = 8):
        super().__init__()
        # Position embedding layer
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        # Shape of the input tensor input_indices: [batch size, sequence length].
        sequence_length = input_indices.shape[1]

        # Create position indices based on the sequence length (example: [0, 1, 2, ..., sequence_length-1])
        position_indices = torch.arange(sequence_length, device=input_indices.device)

        # Get the embedding vectors for the position indices
        position_embeddings = self.position_embedding_layer.forward(position_indices)

        return position_embeddings

class EmbeddingModule(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Embedding layer for each token
        self.token_embedding_layer = TokenEmbedding(vocab_size = vocab_size, embedding_dim = config.embedding_dim)  # Word embedding layer
        self.position_embedding_layer = PositionEmbedding(input_sequence_length = config.input_sequence_length, embedding_dim = config.embedding_dim)  # Embed position information

    def forward(self, input_indices):
        # Get token embeddings
        token_embeddings = self.token_embedding_layer.embed(input_indices)

        # Get position embeddings
        position_embeddings = self.position_embedding_layer.forward(input_indices)

        # Add token embeddings and position embeddings
        embeddings = position_embeddings + token_embeddings
        return embeddings


class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc= nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)

        # Dropout
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape  # Batch, token length, embedding channels

        Key = self.key_fc.forward(input_tensor)     # (B, T, head_size)
        Query = self.query_fc.forward(input_tensor)   # (B, T, head_size)
        Value = self.value_fc.forward(input_tensor)   # (B, T, head_size)

        # Calculate attention scores (QK^T) / sqrt(embedding_dim)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size**(-0.5)

        # Mask applied
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        # Softmax -> dropout -> weighted sum
        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        # Manage multiple heads with ModuleList
        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config)
            for _ in range(self.num_attention_heads)
        ])

        # Linear layer that mixes the outputs of each head
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

        # Dropout for the output
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        # Get the output of each head
        # List of (B, T, head_size)
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]

        # Concatenate the outputs of all heads -> (B, T, embedding_dim)
        concatenated = torch.cat(head_outputs_list, dim=-1)

        # Mix the output with a linear transformation
        projected = self.output_projection.forward(concatenated)

        # Apply dropout to the final output
        output = self.dropout.forward(projected)

        return output

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Each LayerNorm keeps its own beta and gamma.
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)

        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        # The forward method is abbreviated.
        normed_input = self.layer_norm1(input_tensor) # Apply LayerNorm to the input
        attention_output = self.multihead_attention(normed_input) # Apply multi-head attention
        residual_attention = attention_output + input_tensor # Add "before! layernorm1"
        normed_attention = self.layer_norm2(residual_attention) # Apply LayerNorm again to the residual output
        feedforward_output = self.feed_forward(normed_attention) # Apply the feed-forward network
        final_output = feedforward_output + residual_attention # Add "before" layernorm2!

        return final_output

class VocabularyLogits(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Layer normalization
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        # Vocabulary-size projection
        self.vocab_projection = nn.Linear(config.embedding_dim, vocab_size)

    def forward(self, transformer_block_output):
        # Apply Layer normalization to the Transformer block output.
        normalized_output = self.output_norm.forward(transformer_block_output)  # (B, T, C)

        # Convert scores to the vocabulary-size dimension with a linear layer.
        vocab_logits = self.vocab_projection.forward(normalized_output)  # (B, T, V)

        return vocab_logits

class nanoGPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.config = config  # Keep this because it is also used during generation.
        self.embedding = EmbeddingModule(vocab_size, config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(vocab_size=vocab_size, config=config)
        self.criterion = nn.CrossEntropyLoss()

    # Generate text
    def generate(self, input_indices, max_new_tokens):
        # Generate only the specified number of tokens, max_new_tokens
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -self.config.input_sequence_length:] # Crop the input

            # The forward pass returns `(likelihood, loss)`; keep only `likelihood` as `logits`.
            logits, _ = self.forward(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] # Extract the logits of the final token
            probs = F.softmax(last_logits, dim=-1) # Convert likelihoods into probabilities with Softmax

            # Sample the next token
            next_token = torch.multinomial(probs, num_samples=1)

            # Merge the new token and update input_indices.
            input_indices = torch.cat((input_indices, next_token), dim=1)

        # Return the final `input_indices`. Its length is the original `input_indices` plus `max_new_tokens`
        return input_indices

    # Calculate likelihood and loss
    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        # During inference, there is no target, so loss is None
        # Only probabilities (logits) are returned.
        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)

        return logits, loss

#### 🦖 Scale up the model a lot

In this tutorial, **we will turn up the model scale in one big jump.**

| Parameter                   | Before | After      | Ratio | Description                           |
| ----------------------- | --- | -------- | -- | ---------------------------- |
| Embedding dimension (`embedding_dim`) | 64  | **512**  | ×8 | Represents tokens in a higher-dimensional space and boosts expressive power |
| Hidden dimension (`hidden_dim`)      | 256 | **2048** | ×8 | Sets this to 4× the embedding size and expands the model's internal processing power |


Also, we set each head dimension to 64. Since 512 ÷ 64 = 8, the number of attention heads is 8.


In [ ]:
# Config class that stores model settings
class ModelConfig:
    batch_size = 16
    input_sequence_length = 512  # Retrieve longer sequences at once to reduce the share of time spent on data loading.
    total_training_steps = 10_000  # Takes about 1 minute
    device_type = 'cuda'  # Fix the device to GPU
    evaluation_frequency = 100  # Frequency for evaluating model performance
    learning_rate = 0.001  # Learning rate
    evaluation_loops = 10  # Number of loops during evaluation
    ########## NEW ##########
    embedding_dim = 512  # Embedding layer size (number of feature-vector dimensions)
    hidden_dim = 2048
    num_attention_heads = 8  # Number of attention heads
    ########## NEW ##########
    layer_count = 4  # Number of model layers
    dropout_rate = 0.1  # Dropout probability
    random_seed_value = 1337  # Random seed for reproducibility

In [ ]:
# Load the config and set the seed
config = ModelConfig()
torch.manual_seed(config.random_seed_value)  # Set the random seed for reproducibility

In [ ]:
# Load the data
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text_data = f.read()
data_loader = DataLoader(text_data, config)

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

The total number of model parameters increased from about 0.24M to 12.9M,
which is **about 50× larger**.

| Metric     | Before    | After        | Increase      |
| ------ | ------ | ---------- | -------- |
| Number of parameters | About 0.24M | **About 12.9M** | **About 50×** |

By expanding the embeddings and hidden layers,
the model can store and learn more information internally.


In [ ]:
# Display the number of model parameters
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

### 🚀 Training Start!

Now it is finally time to start training.
Because the model is larger, the **compute cost per step is also higher**.

This time, we will use a **15 GB RAM T4 GPU** with
`batch size = 16` and `total steps = 10,000`.

Under these settings, training takes **about 50 minutes** to finish 😇

If you are using an A100 40GB RAM, expect about 15 to 20 minutes.

Honestly, that is a lot of waiting... go have some fun.

Here is a video if you want something to watch 👇 (only 3 minutes)

🎬 [I built ChatGPT with Minecraft redstone!](https://youtu.be/VaeI9YgE1o8?si=8nQ4fQaSx6HP4iWF)


In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

Let’s plot it with `Step` on the horizontal axis and `Loss` on the vertical axis using `matplotlib`.


In [ ]:
# Draw the graph.
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')
plt.plot(trainer.steps, trainer.val_losses, label='Validation Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()

#### The loss plateau went down, but we also see signs of overfitting

In this training run, the validation loss dropped to about 1.5!

Compared with the previous plateau around 1.7 to 1.8, we can clearly see that the loss plateau moved lower.

...but if we look closely, in the second half of training, **the validation loss starts going back up**, and **the gap from the training loss keeps getting wider**.

---

This is a classic pattern of **overfitting**.

---

If we keep training like this, the training loss will get very close to 0, and the model will start reproducing the training data almost exactly.

In other words, it has memorized the text word for word. That is impressive in its own way, though haha.

<br>

The Shakespeare dataset we are using here is still small, with only about 100,000 characters (roughly around 100,000 tokens).

In contrast, we greatly increased the number of model parameters from about 240,000 to about 12 million (about 50×).

This imbalance causes the model to **memorize the data too much**.
On top of that, the total number of tokens processed during training (`total_seen_tokens`) reaches **about 80 million tokens**.
There are only about 100,000 possible token positions for sampling starts, so even though we sample randomly,
we are effectively sampling **the exact same text hundreds of times**.


In [ ]:
# Try inference too
# Switch to evaluation mode. Disable dropout.
model.eval()
print("Model set to eval mode")

In [ ]:
prompt = "Let's he"
print(f"\nInput prompt: {prompt}")

# Tokenize and convert to a tensor
encoded = data_loader.encode(prompt) # Encode the text into IDs
print("encoded", encoded)
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # Convert the list of IDs into tensor format
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.unsqueeze(0)  # Add a batch dimension
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.to(config.device_type) # Transfer encoded_tensor to cuda (GPU)
print_formatted_tensor(encoded_tensor)

In [ ]:
# Generate text
generated_text = model.generate(encoded_tensor, max_new_tokens=512)

In [ ]:
decoded_text = data_loader.decode(generated_text[0].tolist())
print(decoded_text)

The spelling is almost perfect. Since this is an overfit model, the output lacks variety, though.


**Section 2: Overfitting in LLMs** <label><input type="checkbox"> Mark as Done</label>


### **Section 3: Saving to Google Drive**


This time, we are using the T4 GPU heavily for training, so let’s store the training logs properly.

Google Drive is the recommended choice.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Collect logs from the trained trainer
results = {
    "step": trainer.steps,
    "train_loss": trainer.train_losses,
    "val_loss": trainer.val_losses,
    "total_seen_tokens": trainer.total_seen_tokens_list,
    "total_train_time": trainer.total_train_time_list,
}

print(results)

```python
function: pd.DataFrame
arguments: results (a data structure such as a dictionary or list)
```



In [ ]:
import pandas as pd
# Convert to a pandas DataFrame
df = # TODO: function(arguments)

In [ ]:
df

```python
function: os.makedirs
arguments: dir_path, exist_ok=True
```

* `os.makedirs(dir_path)` creates a directory at the specified path.
* By setting `exist_ok=True`, it skips without an error even if the folder already exists.


In [ ]:
# Create the destination folder
import os
dir_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter16"
# TODO: function(arguments)

In [ ]:
# Specify the destination path for saving as a CSV file.
save_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter16/training_logs.csv"

```python
instance: df (a pandas DataFrame)
method: to_csv
arguments: save_path, index=False
```

* `df.to_csv(save_path, index=False)` saves the contents of `df` as a CSV file.
* If you set `index=False`, the DataFrame row numbers (index) are not written to the CSV.


In [ ]:
# Save as CSV
# TODO: instance.method(arguments)
print(f"✅ CSV saved to: {save_path}")

#### 📂 How to Check the Log File

Open [Google Drive](https://drive.google.com/drive/u/0/home) and check it there.

A CSV log file should have been created inside the folder
**`nanoGPT_logs/Chapter16`**.

---

⚠️ **Note:**
If you continue after Google Drive failed to mount,
the folders and files will be saved in a folder named `drive` inside **Colab's local environment**. They will not be on Google Drive.

In that case, mount Drive again.

---



**Check Point** <label><input type="checkbox">Confirmed the CSV file on Google Drive<label>


Next, let’s also save the `config` instance that stores the model information as `config.json`.


```python
function: vars
arguments: config.__class__
```

* `vars(config.__class__)` gets the attributes held by the class of the `config` instance (`config.__class__`) as a dictionary.
* `vars` is the recommended built-in function for accessing `__dict__`.
* As a result, you get a dictionary in the form `{attribute_name: attribute_value, ...}`.


In [ ]:
# Convert class attributes to a dictionary
config_class_dict = # TODO: function(arguments)
print(config_class_dict)

```python
original varaible: config_class_dict
method: items
arguments: none

new varaible = original varaible.method(arguments)
```

* Using `.items()` gives us a form where we can process each class attribute name (key) and its value one by one.


In [ ]:
# Get the dictionary (key, value) pairs
config_dict_items = # TODO: new varaible = original varaible.method(arguments)
print(config_dict_items)

There is an extra key called `__module__`, so we will remove it.

Writing it from scratch is a bit much, so you only need to type it in.


```python
config_dict = {
    key: value
    for key, value in config_dict_items
    if not key.startswith("__")
}
```

In [ ]:
config_dict = # TODO:

print(config_dict)

```python
function: os.path.join
arguments: dir_path, "model_config.json"
```

* `os.path.join()` combines multiple path parts and creates a file path that works across platforms.
* For example, it correctly builds paths using `\` on Windows and `/` on Unix-like systems.
* In this case, it creates the path to the `"model_config.json"` file inside the `dir_path` directory.


In [ ]:
# Create the destination file path.
# dir_path = "/content/drive/MyDrive/nanoGPT_logs/Chapter16"
config_path = # TODO: function(arguments)
print(config_path)

```python
function: json.dump
arguments: config_dict, f
```

* Write `config_dict` to the file `f` in JSON format.


In [ ]:
import json
# Open the file in write mode (w) so it can be handled with the variable `f`
with open(config_path, "w") as f:
    # TODO: function(arguments)

print(f"✅ Config saved to: {config_path}")

**Section 3: Saving to Google Drive** <label><input type="checkbox"> Mark as Done</label>


**⚠️ Use the 🔽 in the top-right corner to disconnect the runtime and stop using credits.** <label><input type="checkbox">Disconnected</label>


**`Chapter 16: Increase Only the Model Size`** <label><input type="checkbox"> Mark as Done</label>
